In [7]:
import pandas as pd
import random as rand
import openpyxl

# Function to simulate rolling dice
def dice(amount:int,sides:int):
    roll = 0
    for i in range(0,amount):
        die = rand.randint(1,sides)
        roll = roll + die
    return(roll)

# Function to find a specific character in a string
# Returns location or None if that character is not in the string
def string_finder(string:str,char:str):
    for pos in range(len(string)):
        if string[pos] == char:
            return(pos)
        elif pos == len(string):
            return(False)  

# Function to find the row index of a die roll within a table of ranges
# Returns column index within given column constraints    
def range_finder(roll_result:int,table_name:str,column_index:int,start_index:int,end_index:int):
    odds_list = table_name.iloc[start_index:end_index,column_index].tolist()
    for row in range(len(odds_list)):
        odds = odds_list[row]
        hyphen_loc = string_finder(odds,'-')
        
        if hyphen_loc == None:
            low = int(odds)
            high = low
        else:
            low = int(odds[0:hyphen_loc])
            high = int(odds[hyphen_loc+1:len(odds)])

        if low <= roll_result <= high:
            return row
      

ModuleNotFoundError: No module named 'pandas'

In [2]:
CR = rand.randint(1,20)
CR

20

In [15]:
# Obtain individual encounter tables from Excel sheet
indv_tables = pd.read_excel('C:/Users/Anderson/Documents/DnD_data.xlsx',sheet_name='Individual Treasure Tables')

# Find indicies where all values are empty (these separate the tables on the excel sheet)
sep = indv_tables[indv_tables.isnull().all(axis=1)].index.tolist()

# Determine the range of indicies to pull from the table based the encounter's Challenge Rating
if 0 <= CR <= 4:
    start = 0
    end = sep[0]
elif 5 <= CR <= 10:
    start = sep[0]+1
    end = sep[1]
elif 11 <= CR <= 16:
    start = sep[1]+1
    end = sep[2]
elif CR > 16:
    start = sep[2]+1
    end = len(indv_tables)
else:
    print("Error: Please enter a positive CR.")

# Extract appropriate indicies from encounter table
loot_table = indv_tables.iloc[start:end].reset_index(drop=True)
loot_table

,d100,cp,sp,ep,gp,pp
0,1-30,5d6,NaN,NaN,NaN,NaN
1,31-60,NaN,4d6,NaN,NaN,NaN
2,61-70,NaN,NaN,3d6,NaN,NaN
3,71-95,NaN,NaN,NaN,3d6,NaN
4,96-100,NaN,NaN,NaN,NaN,1d6


In [6]:
# Roll a d100 to determine the row of the loot table to use
loot_roll = dice(1,100)
loot_index = range_finder(loot_roll,loot_table,0,0,len(loot_table))
print("roll:",loot_roll)
print("row:",loot_index)


roll: 95
row: 3


In [7]:
# Drop empty cells from loot table
coin_table = loot_table.loc[loot_index,'cp':].dropna()
for coin in coin_table:
    # Find location of 'd' and ' ' within the string
    d_loc = string_finder(coin,'d')
    space_loc = string_finder(coin,' ')

    # Extract the amount of dice to roll
    amt = int(coin[0:d_loc])

    # If there is no ' ' in the string (no multiplier)
    if space_loc == None:
        # Extract the size of dice to roll
        sides = int(coin[d_loc+1:])
        value = dice(amt,sides)
    # If the string has a multiplier attatched
    else:
        sides = int(coin[d_loc+1:space_loc])
        mult = int((coin[space_loc+3:]))
        value = dice(amt,sides)*mult
    
    # Extract the denomination of coin from loot table
    denom = coin_table.index[coin_table==coin][0] # for some reason this creates a list with the string and type so just pull the first part
    print(value,denom)

400 gp
60 pp


In [4]:


# Import Excel sheet containing treasure hoard data
hoard_tables = pd.read_excel('C:/Users/Anderson/OneDrive/Documents/Python/DnD_Data.xlsx',sheet_name='Treasure Hoard Tables',header=None)

# Find indicies where all values are empty (these separate the tables on the excel sheet)
sep = hoard_tables[hoard_tables.isnull().all(axis=1)].index.tolist()

# Determine which table to use based on Challenge Rating
if 0 <= CR <= 4:
    loot_table = hoard_tables.iloc[0:sep[0]]
elif 5 <= CR <= 10:
    loot_table = hoard_tables.iloc[sep[0]+1:sep[1]]
elif 11 <= CR <= 16:
    loot_table = hoard_tables.iloc[sep[1]+1:sep[2]]
elif CR > 16:
    loot_table = hoard_tables.iloc[sep[2]+1:len(hoard_tables)]
else:
    print("Error: Please enter a positive CR.")

# Set table to base index values
loot_table = loot_table.reset_index(drop=True)
loot_table

,0,1,2,3,4,5
0,,cp,sp,ep,gp,pp
1,coins,NaN,NaN,NaN,12d6 x 1000,8d6 x 1000
2,d100,Gems or Art Objects,Magic Items,NaN,NaN,NaN
3,1-2,NaN,NaN,NaN,NaN,NaN
4,3-5,3d6 1000 gp gems,Roll 1d8 times on Magic Item Table C,NaN,NaN,NaN
5,6-8,1d10 2500 gp art objects,Roll 1d8 times on Magic Item Table C,NaN,NaN,NaN
6,9-11,1d4 7500 gp art objects,Roll 1d8 times on Magic Item Table C,NaN,NaN,NaN
7,12-14,1d8 5000 gp gems,Roll 1d8 times on Magic Item Table C,NaN,NaN,NaN
8,15-22,3d6 1000 gp gems,Roll 1d6 times on Magic Item Table D,NaN,NaN,NaN
9,23-30,1d10 2500 gp art objects,Roll 1d6 times on Magic Item Table D,NaN,NaN,NaN


In [7]:
for coin_index in loot_table.iloc[1:2,1:].dropna(axis=1):
    coins = loot_table.iloc[1,coin_index]
    # Find location of 'd' and ' ' within the string
    d_loc = string_finder(coins,'d')
    space_loc = string_finder(coins,' ')

    # Extract the amount of dice to roll
    amt = int(coins[0:d_loc])

    # If there is no ' ' in the string (no multiplier)
    if space_loc == None:
        # Extract the size of dice to roll
        sides = int(coins[d_loc+1:])
        value = dice(amt,sides)
    # If the string has a multiplier attatched
    else:
        sides = int(coins[d_loc+1:space_loc])
        mult = int((coins[space_loc+3:]))
        value = dice(amt,sides)*mult
    
    # Extract the denomination of coin from loot table
    denom = loot_table.iloc[0,coin_index] # for some reason this creates a list with the string and type so just pull the first part
    print(value,denom)

36000 gp
31000 pp


In [10]:
items_table = loot_table.iloc[3:len(loot_table),0:2].reset_index(drop=True)
items_table

,0,1
0,1-2,NaN
1,3-5,3d6 1000 gp gems
2,6-8,1d10 2500 gp art objects
3,9-11,1d4 7500 gp art objects
4,12-14,1d8 5000 gp gems
5,15-22,3d6 1000 gp gems
6,23-30,1d10 2500 gp art objects
7,31-38,1d4 7500 gp art objects
8,39-46,1d8 5000 gp gems
9,47-52,3d6 1000 gp gems


In [11]:
loot_roll = dice(1,100)
treasure_index = range_finder(loot_roll,items_table,0,0,len(items_table))

if pd.isna(items_table.iloc[treasure_index,1]) == False:
    print('h')

h


In [12]:
# Isolate the gems/art objects column
valuables = items_table.iloc[treasure_index,1]

# There's a chance that the space is empty in this case so check for that
if pd.isna(valuables) == False:

    # Find the location of "d" and the first space
    d_loc = string_finder(valuables,'d')
    space_loc = string_finder(valuables,' ')

    # Read the number and type of dice and the type of valuables
    amt = int(valuables[0:d_loc])
    sides = int(valuables[d_loc+1:space_loc])
    val_type = valuables[space_loc+1:len(valuables)]

    # Roll the number of gems or art objects in the hoard
    val_amt = dice(amt,sides)
    print(val_amt,val_type)

3 7500 gp art objects


In [14]:
# Constrain loot table to just the magic items section
magic_table = loot_table.iloc[3:,2:].reset_index(drop=True)

# Grab the row according to the index already determined by the gems/art objects
magic_row = magic_table.iloc[treasure_index,:]

# Obtain the Magic Item Tables from the Excel file
magic_items = pd.read_excel('C:/Users/Anderson/OneDrive/Documents/Python/DnD_Data.xlsx',sheet_name='Magic Item Tables',index_col=0)
magic_table

,2,3,4,5
0,NaN,NaN,NaN,NaN
1,Roll 1d8 times on Magic Item Table C,NaN,NaN,NaN
2,Roll 1d8 times on Magic Item Table C,NaN,NaN,NaN
3,Roll 1d8 times on Magic Item Table C,NaN,NaN,NaN
4,Roll 1d8 times on Magic Item Table C,NaN,NaN,NaN
5,Roll 1d6 times on Magic Item Table D,NaN,NaN,NaN
6,Roll 1d6 times on Magic Item Table D,NaN,NaN,NaN
7,Roll 1d6 times on Magic Item Table D,NaN,NaN,NaN
8,Roll 1d6 times on Magic Item Table D,NaN,NaN,NaN
9,Roll 1d6 times on Magic Item Table E,NaN,NaN,NaN


In [14]:
items = []
items

[]

In [124]:
# Preallocate a list to put the items into
items = []
# For each column that magic items could be in
for column in range(4):
    
    # Extract instructions for which table to use
    description = magic_table.iloc[treasure_index,column]

    # Detect if the index contains information or not
    NaN = magic_table.iloc[treasure_index].notna()
    
    # Only try to do something if there's something there :)
    if NaN.iloc[column] == 1:
        # Check if just one roll ("Roll once")
        if description[5] == 'o':
            number_items = 1
        
        # Determine the number of items for each column
        else:
            d_loc = string_finder(description,'d')
            t_loc = string_finder(description,'t')

            amt = int(description[5:d_loc])
            sides = int(description[d_loc+1:t_loc-1])
            number_items = dice(amt,sides)
        
        # The particular table is at the end of the string so just index this
        table = description[len(description)-1]
        # Obtain the letter magic item table
        crnt_magic_table = magic_items.loc[table].reset_index(drop=True)
        
        # Repeat for the number of items rolled
        for i in range(number_items):
            dupe = False
            item_roll = dice(1,100) 
            # Identify the particular item from the magic item table
            item_index = range_finder(item_roll,crnt_magic_table,0,0,len(crnt_magic_table))
            # Get item from table at determined index
            item = crnt_magic_table.loc[item_index,'Name']

            # Duplicate item check to display as x2 or x3 etc
            for check in items:
                # Check if the list index already has a multiplier
                x_loc = string_finder(check,'x')
                
                # Function returns None if the character is not present
                if x_loc != None:
                    # Determine the current count based on the number after the x
                    crnt_count = int(check[x_loc+1:])
                    # Reset to the string without the multiplier & space
                    base_string = check[:x_loc-1]

                else:
                    crnt_count = 1
                    base_string = check

                # Check if the rolled item is the same as the index
                if item == base_string:
                    # Update the count by one
                    crnt_count = crnt_count+1
                    # Replace the item in it's place in the list
                    items[items.index(check)] = base_string + ' x' + str(crnt_count)
                    # Not necessary to append to the list if this is the case
                    dupe = True
            # Only need to append list if item was not replaced
            if dupe == False:
                items.append(item)

# Display every item in the list
for item in items:
    print(item)


Potion of Supreme Healing x2
Spell Scroll (7th)
Potion of Flying
Potion of Speed


In [5]:
h = (1,2,3,4,5)
h[-1]

5

In [2]:
import pandas as pd
data = pd.read_excel('C:/Users/Anderson/Documents/DnD_data.xlsx',sheet_name='Magic Items List',header=None)
items = data.iloc[:,0].tolist()
items

['Adamantine Breastplate',
 'Adamantine Chain Mail',
 'Adamantine Chain Shirt',
 'Adamantine Half Plate',
 'Adamantine Plate Armor',
 'Adamantine Scale Mail',
 'Adamantine Splint Armor',
 'Alchemy Jug',
 'Ammunition +1',
 'Ammunition +2',
 'Ammunition +3',
 'Amulet of Health',
 'Amulet of Proof Against Detection and Location',
 'Amulet of the Planes',
 'Animated Shield',
 'Apparatus of Kwalish',
 'Armor of Invulnerability',
 'Armor of Resistance (breastplate)',
 'Armor of Resistance (chain mail)',
 'Armor of Resistance (chain shirt)',
 'Armor of Resistance (half plate)',
 'Armor of Resistance (leather)',
 'Armor of Resistance (scale mail)',
 'Armor of Resistance (splint)',
 'Armor of Resistance (studded leather)',
 'Armor of Vulnerability',
 'Arrow of Slaying',
 'Arrow-Catching Shield',
 'Bag of Beans',
 'Bag of Devouring',
 'Bag of Holding',
 'Bag of Tricks (gray)',
 'Bag of Tricks (rust)',
 'Bag of Tricks (tan)',
 'Bead of Force',
 'Belt of Cloud Giant Strenght',
 'Belt of Dwarvenkin

In [34]:
text = "a"
text = text.casefold()
# create new list
new_items = []
# index every item in existing list
for caps_item in items:
    # ignore casing
    item = caps_item.casefold()
    # initialize boolean
    match = False
    # index for every letter in item name
    for item_index in range(len(item)):
        # set the starting letter we are checking if same
        check_letter = item[item_index]
        text_index = 0
        # check if series of letters match beginning at starting letter
        while item[item_index+text_index] == text[text_index]:
            # if we successfully matched the entire text prompt
            if text_index == len(text)-1:
                match = True
                break

            elif item_index+text_index == len(item)-1:
                break
            # go to next letter in text prompt
            text_index = text_index+1

        # check if the item matches search
        if match == True:
            # add to new list of items
            new_items = new_items + [caps_item]
            break
print(new_items)

# save original index and use that to get original capitalized version of item (use dataframe instead of list? or just use the index of "item" in first for loop)

['Adamantine Breastplate', 'Adamantine Chain Mail', 'Adamantine Chain Shirt', 'Adamantine Half Plate', 'Adamantine Plate Armor', 'Adamantine Scale Mail', 'Adamantine Splint Armor', 'Alchemy Jug', 'Ammunition +1', 'Ammunition +2', 'Ammunition +3', 'Amulet of Health', 'Amulet of Proof Against Detection and Location', 'Amulet of the Planes', 'Animated Shield', 'Apparatus of Kwalish', 'Armor of Invulnerability', 'Armor of Resistance (breastplate)', 'Armor of Resistance (chain mail)', 'Armor of Resistance (chain shirt)', 'Armor of Resistance (half plate)', 'Armor of Resistance (leather)', 'Armor of Resistance (scale mail)', 'Armor of Resistance (splint)', 'Armor of Resistance (studded leather)', 'Armor of Vulnerability', 'Arrow of Slaying', 'Arrow-Catching Shield', 'Bag of Beans', 'Bag of Devouring', 'Bag of Holding', 'Bag of Tricks (gray)', 'Bag of Tricks (rust)', 'Bag of Tricks (tan)', 'Bead of Force', 'Belt of Cloud Giant Strenght', 'Belt of Dwarvenkind', 'Belt of Fire Giant Strength', '

In [11]:
hoard_tables = pd.read_excel('C:/Users/Anderson/Documents/DnD_data.xlsx',sheet_name='Treasure Hoard Tables',header=None)
hoard_tables.iloc[0:30,:]

,0,1,2,3,4,5
0,,cp,sp,ep,gp,pp
1,coins,6d6 x 100,3d6 x 100,NaN,2d6 x 10,NaN
2,d100,Gems or Art Objects,Magic Items,NaN,NaN,NaN
3,1-6,NaN,NaN,NaN,NaN,NaN
4,7-16,2d6 10 gp gems,NaN,NaN,NaN,NaN
5,17-26,2d4 25 gp art objects,NaN,NaN,NaN,NaN
6,27-36,2d6 50 gp gems,NaN,NaN,NaN,NaN
7,37-44,2d6 10 gp gems,Roll 1d6 times on Magic Item Table A,NaN,NaN,NaN
8,45-52,2d4 25 gp art objects,Roll 1d6 times on Magic Item Table B,NaN,NaN,NaN
9,53-60,2d6 50 gp gems,Roll 1d6 times on Magic Item Table B,NaN,NaN,NaN


In [32]:
import pandas as pd
import numpy as np
import openpyxl

def string_finder(string:str,char:str):
    for pos in range(len(string)):
        if string[pos] == char:
            return(pos)
        elif pos == len(string):
            return(False)

indv_tables = pd.read_excel('C:/Users/Anderson/Documents/DnD_data.xlsx',sheet_name='Individual Treasure Tables')

i = 6
for coin_type in indv_tables.columns[1:]:
    indv_tables.rename(columns={coin_type:coin_type+"_low"},inplace=True)
    indv_tables.insert(i,coin_type+"_high",np.nan)
    i = i-1

indv_tables.rename(columns={"d100":"low"},inplace=True)
indv_tables.insert(1,"high",np.nan)

for index in indv_tables.index:
    dice_range = indv_tables.loc[index,"low"]
    if pd.notna(dice_range)==True:
        hyphen_loc = string_finder(dice_range,'-')
        low = dice_range[0:hyphen_loc]
        high = dice_range[hyphen_loc+1:]

        indv_tables.loc[index,"low"] = low
        indv_tables.loc[index,"high"] = high

    for coin_type in indv_tables.columns[1:]:
        loot_dice = indv_tables.loc[index,coin_type]
        print(loot_dice)
        if pd.notna(loot_dice)==True:
            d_loc = string_finder(loot_dice,'d')
            x_loc = string_finder(loot_dice,'x')

            print(d_loc)

            dice_amount = loot_dice[0:d_loc]

            if x_loc:
                dice_size = loot_dice[d_loc+1,x_loc-1]
                dice_multiplier = loot_dice[x_loc+1,:]

            else:
                dice_size = loot_dice[d_loc+1,:]
                dice_multiplier = 1
indv_tables

30
None


TypeError: unsupported operand type(s) for +: 'NoneType' and 'int'

In [46]:
indv_tables

,low,high,cp,sp,ep,gp,pp
0,1-30,NaN,5d6,NaN,NaN,NaN,NaN
1,31-60,NaN,NaN,4d6,NaN,NaN,NaN
2,61-70,NaN,NaN,NaN,3d6,NaN,NaN
3,71-95,NaN,NaN,NaN,NaN,3d6,NaN
4,96-100,NaN,NaN,NaN,NaN,NaN,1d6
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,1-30,NaN,4d6 x 100,NaN,1d6 x 10,NaN,NaN
7,31-60,NaN,NaN,6d6 x 10,NaN,2d6 x 10,NaN
8,61-70,NaN,NaN,NaN,3d6 x 10,2d6 x 10,NaN
9,71-95,NaN,NaN,NaN,NaN,4d6 x 10,NaN


In [2]:
import pandas as pd
import numpy as np
import openpyxl

def string_finder(string:str,char:str):
    for pos in range(len(string)):
        if string[pos] == char:
            return(pos)
        elif pos == len(string):
            return(False)

indv_tables = pd.read_excel('C:/Users/Anderson/Documents/DnD_data.xlsx',sheet_name='Individual Treasure Tables')
new_tables = indv_tables.copy()

new_tables.rename(columns={"d100":"low"},inplace=True)
new_tables.insert(1,"high",np.nan)

insert_index = [i for i in range(3,16,3)]
label_list = ["_amt","_size","_mult"]

for coin_num in range(0,5):
    coin_name = indv_tables.columns[1+coin_num]

    for label_num in range(0,3):
        label = label_list[label_num]
        column_num = insert_index[coin_num]+label_num

        new_tables.insert(column_num,coin_name+label,np.nan)

    loot_column = indv_tables.loc[:,coin_name].tolist()
    for loot_index in range(len(loot_column)):
        loot_dice = loot_column[loot_index]
        if pd.notna(loot_dice)==True:
            d_loc = string_finder(loot_dice,'d')
            x_loc = string_finder(loot_dice,'x')

            amt = loot_dice[0:d_loc]

            if x_loc:
                size = loot_dice[d_loc+1:x_loc-1]
                mult = loot_dice[x_loc+1:]

            else:
                size = loot_dice[d_loc+1:]
                mult = 1

            dice_info = [amt,size,mult]
            for label_num in range(0,3):
                new_tables.loc[loot_index,coin_name+label_list[label_num]] = dice_info[label_num]

    del new_tables[coin_name]


new_tables
            

        

FileNotFoundError: [Errno 2] No such file or directory: 'C:/Users/Anderson/Documents/DnD_data.xlsx'

In [1]:
[str(i) for i in range(17)]+['17+']


['0',
 '1',
 '2',
 '3',
 '4',
 '5',
 '6',
 '7',
 '8',
 '9',
 '10',
 '11',
 '12',
 '13',
 '14',
 '15',
 '16',
 '17+']

In [3]:
import pandas as pd
app_info = pd.read_csv("App Info.csv")
try:
    file_path = app_info.loc[0,"File Path"]
    pd.read_excel(file_path)
    print(file_path)
except (KeyError,ValueError):
    print('h')

previous_items = app_info.loc[:,"campain"]
previous_items[0]

C:/Users/Anderson/OneDrive/Documents/Python/DnD_Data.xlsx


np.float64(nan)

In [58]:
import tkinter as tk
from tkinter import ttk
import pandas as pd
import random as rand
import openpyxl

def center_screen(window,window_width:int,window_height:int):
    screen_width = window.winfo_screenwidth()
    screen_height = window.winfo_screenheight()
    x = (screen_width // 2) - (window_width // 2)
    y = (screen_height // 2) - (window_height // 2)
    window_geometry = f"{window_width}x{window_height}+{x}+{y}"
    return window_geometry

try:
    app_info = pd.read_csv("App Info.csv")
    pd.read_excel(app_info.loc[0,"File Path"])

except (FileNotFoundError,KeyError,ValueError):
    # Create a new window with a text prompt
    get_path = tk.Tk()
    get_path.title("Locate Data")
    get_path.geometry(center_screen(get_path,210,50))

    # Save the user input in the txt file
    def insert_path(event=None):
        text = path_entry.get()
        # Build a new string for the file path letter by letter
        file_path = ""
        for letter in text:
            # Convert backslash from windows "copy path" command to forward slash that python uses
            if letter == "\\":
                file_path = file_path + "/"
            # Ignore quotation marks
            elif letter != "\"":
                file_path = file_path + letter
        
        # Check to make sure input works
        try:
            pd.read_excel(file_path)
            app_info = pd.DataFrame({"File Path":[file_path]})
            app_info.to_csv("App Info.csv",index=None)
            # Close window when successful
            get_path.destroy()
            
        except FileNotFoundError:
            instruct.config(text="File name not found. Try again, stupid.")

    get_path.rowconfigure(0,weight=1)
    get_path.columnconfigure(0,weight=1)
   
    center = tk.Frame(get_path)
    center.grid(row=0,column=0)

    instructions = "Enter the file path of the item tables."
    instruct = ttk.Label(center,text=instructions)
    instruct.grid(row=0,column=0,columnspan=2)

    path_entry = ttk.Entry(center)
    path_entry.grid(row=1,column=0,padx=3)
    path_entry.bind("<Return>",insert_path)

    insert_btn = ttk.Button(center,text="Insert",command=insert_path)
    insert_btn.grid(row=1,column=1)

    get_path.mainloop()

app_info = pd.read_csv("App Info.csv")
file_path = app_info.loc[0,"File Path"]
print(app_info)

                                           File Path   a   b   c
0  C:/Users/Anderson/OneDrive/Documents/Python/Dn... NaN NaN NaN


In [59]:
camp_name = 'a'
new_column = pd.concat([app_info.loc[:,camp_name].dropna(ignore_index=True),pd.Series(['item 31','item 32'])],ignore_index=True)
if len(new_column) > len(app_info):
    app_info = app_info.reindex(range(len(new_column)))

elif len (new_column) < len(app_info):
    new_column = new_column.reindex(range(len(app_info)))

app_info[camp_name] = new_column
print(app_info)

                                           File Path        a   b   c
0  C:/Users/Anderson/OneDrive/Documents/Python/Dn...  item 31 NaN NaN
1                                                NaN  item 32 NaN NaN
